# MLIS II 2026 — CNN Steering Angle & Speed Prediction

- Task is to predict steering angle (regession) which is normalised to be between 0 and 1, and speed (binary classification) which is either 0 or 1
- Number of valid training samples are 14,383 and each are 320x240 RGB images 
- 0 count is 2944, 1 is 11439 --> significant class imbalance
- Using validation set for early stopping and test for model comparison (80/10/10 split)

## Model 1- Architecture
| Layer | Details |
|---|---|
| Input | 224 × 224 × 3 |
| Conv Block 1 | 32 filters, 3×3, ReLU, MaxPool |
| Conv Block 2 | 64 filters, 3×3, ReLU, MaxPool |
| Conv Block 3 | 128 filters, 3×3, ReLU, MaxPool |
| Flatten | — |
| Dense 1 | 128 units, ReLU |
| Output: angle | 1 unit, Sigmoid |
| Output: speed | 1 unit, Sigmoid |

### Hyperparameters

| Parameter | Value |
|---|---|
| Optimizer | Adam |
| Initial LR | 1e-3 |
| LR schedule | ReduceLROnPlateau (factor=0.5, patience=3) |
| Batch size | 16 |
| Angle loss | MSE |
| Speed loss | Weighted BCE (w0=2.4, w1=0.6) |
| Early stopping | patience=7 |

### Results

![Angle MSE](images/model2_angle_mse.png)
![Speed MSE](images/model2_speed_mse.png)

Test Metrics; 
Combined MSE:  0.0368,
Angle MSE:      0.0119,
Speed MSE:      0.0617

## Model 2; Using Transfer Learning

### Hyperparameters

| Parameter | Value |
|---|---|
| Backbone | EfficientNetB0 (ImageNet) |
| Unfrozen layers | 10 |
| Dense units | 256 |
| Dropout | 0.5 |
| Optimizer | Adam |
| Initial LR | 5e-5 |
| LR schedule | ReduceLROnPlateau (factor=0.5, patience=3) |
| Batch size | 16 |
| Angle loss | MSE |
| Speed loss | Weighted BCE (w0=2.4, w1=0.6) |
| Early stopping | patience=7 |

- Worse results than model 1, suggesting that this task is too domain-specific
- Will improve model 1 architecture in next iteration

## Model 3- Architecture

| Layer | Details |
|---|---|
| Input | 224 × 224 × 3 |
| Conv Block 1 | 32 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Conv Block 2 | 64 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Conv Block 3 | 128 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Conv Block 4 | 256 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Pooling | GlobalAveragePooling2D |
| Dense 1 | 256 units, ReLU, Dropout 0.4 |
| Dense 2 | 128 units, ReLU, Dropout 0.4 |
| Output: angle | 1 unit, Sigmoid |
| Output: speed | 1 unit, Sigmoid |

- Uses GlobalAveragePooling2D instead of flatten layer --> much smaller vector output --> less parameters and overfitting
- Dropout in dense layers to reduce overfitting 
- BatchNorm normalises the output of each layer so it has mean≈0 and std≈1

### Results

![Angle MSE](images/model3_angle_mse.png)
![Speed MSE](images/model3_speed_mse.png)

Test Metrics;
Combined MSE:  0.0239,
Angle MSE:      0.0120,
Speed MSE:      0.0358

## Model 4 - Architecture

- Doubling the filters at each convolution layer --> can learn more complex features

| Layer | Details |
|---|---|
| Input | 224 × 224 × 3 |
| Conv Block 1 | 64 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Conv Block 2 | 128 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Conv Block 3 | 256 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Conv Block 4 | 512 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Pooling | GlobalAveragePooling2D |
| Dense 1 | 256 units, ReLU, Dropout 0.4 |
| Dense 2 | 128 units, ReLU, Dropout 0.4 |
| Output: angle | 1 unit, Sigmoid |
| Output: speed | 1 unit, Sigmoid |

### Results

![Angle MSE](images/model4_angle_mse.png)
![Speed MSE](images/model4_speed_mse.png)

Test Metrics;
Combined MSE:  0.0219,
Angle MSE:      0.0128,
Speed MSE:      0.0309

## Model 5; 2 Separate CNN Models

### Angle CNN- Architecture

| Layer | Details |
|---|---|
| Input | 224 × 224 × 3 |
| Conv Block 1 | 64 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Conv Block 2 | 128 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Conv Block 3 | 256 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Conv Block 4 | 512 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Pooling | GlobalAveragePooling2D |
| Dense 1 | 256 units, ReLU, Dropout 0.3 |
| Dense 2 | 128 units, ReLU, Dropout 0.3 |
| Output | 1 unit, Sigmoid |

### Speed CNN- Architecture
| Layer | Details |
|---|---|
| Input | 224 × 224 × 3 |
| Conv Block 1 | 32 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Conv Block 2 | 64 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Conv Block 3 | 128 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Conv Block 4 | 256 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Pooling | GlobalAveragePooling2D |
| Dense 1 | 128 units, ReLU, Dropout 0.5 |
| Output | 1 unit, Sigmoid |

### Results

![Angle MSE](images/model5_angle_mse.png)
![Speed MSE](images/model5_speed_mse.png)

Test Metrics;
Combined MSE:  0.01177,
Angle MSE:      0.00840,
Speed MSE:      0.01177

- More effective to have two separate architectures 
- Next steps: apply data augmentation, use transfer learning on speed CNN, add more filters to convolution layers in speed CNN


## Model 6

- Adding data augmentation layers within angle CNN model
- Expanding speed CNN architecture

### Angle CNN- Architecture

| Layer | Details |
|---|---|
| Input | 224 × 224 × 3 |
| RandomBrightness | factor=0.1 |
| RandomContrast | factor=0.1 |
| Conv Block 1 | 64 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Conv Block 2 | 128 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Conv Block 3 | 256 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Conv Block 4 | 512 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Pooling | GlobalAveragePooling2D |
| Dense 1 | 256 units, ReLU, Dropout 0.3 |
| Dense 2 | 128 units, ReLU, Dropout 0.3 |
| Output | 1 unit, Sigmoid |

### Speed CNN- Architecture

| Layer | Details |
|---|---|
| Input | 224 × 224 × 3 |
| Conv Block 1 | 64 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Conv Block 2 | 128 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Conv Block 3 | 256 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Conv Block 4 | 512 filters, 3×3, BatchNorm, ReLU, MaxPool |
| Pooling | GlobalAveragePooling2D |
| Dense 1 | 256 units, ReLU, Dropout 0.4 |
| Dense 2 | 128 units, ReLU, Dropout 0.4 |
| Output | 1 unit, Sigmoid |

### Results

![Angle MSE](images/model6_angle_mse.png)
![Speed MSE](images/model6_speed_mse.png)

Test Metrics;
Combined MSE:  0.01353,
Angle MSE:      0.00792,
Speed MSE:      0.01913

## Model 7; Adding Transfer Learning